# 02 — Statistical analysis

Trend, volatility, returns, seasonality and correlation — and what each one implies for the modelling choices made later.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.io as pio

pio.renderers.default = "notebook_connected"  # CDN reference, not an embedded copy of plotly.js per figure
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from mru_forecast import viz
from mru_forecast.config import load_config
from mru_forecast.data.cleaning import load_processed

config = load_config()
frame = load_processed(config)
TARGETS = [t["name"] for t in config["targets"]]
print(f"{len(frame)} business days: {frame.index.min().date()} to {frame.index.max().date()}")
frame.tail(3).round(4)


2053 business days: 2018-11-01 to 2026-09-14


,BRENT_OIL,EUR_MRU,EUR_USD,GOLD,USD_MRU
date,,,,,
2026-09-10,104.715,46.0719,1.1611,4391.7000,39.6801
2026-09-11,106.885,46.0719,1.1611,4384.1499,39.6801
2026-09-14,106.360,46.2481,1.1553,4338.7000,40.0300


## 1. Stationarity

ADF and KPSS test opposite null hypotheses. Agreement between them is what makes a conclusion credible.

In [2]:
from mru_forecast.evaluation import eda

returns = eda.compute_returns(frame, "log")
rows = []
for name in TARGETS:
    for label, series in (("level", frame[name]), ("log-return", returns[name])):
        table = eda.stationarity_tests(series)
        table.insert(0, "series", name); table.insert(1, "transform", label)
        rows.append(table)
pd.concat(rows, ignore_index=True)[
    ["series", "transform", "test", "statistic", "p_value", "interpretation"]
].round(4)

,series,transform,test,statistic,p_value,interpretation
0,USD_MRU,level,ADF,-1.2810,0.6378,non-stationary
1,USD_MRU,level,KPSS,3.9920,0.0100,non-stationary
2,USD_MRU,log-return,ADF,-10.0112,0.0000,stationary
3,USD_MRU,log-return,KPSS,0.0575,0.1000,stationary
4,EUR_MRU,level,ADF,-1.4840,0.5414,non-stationary
5,EUR_MRU,level,KPSS,1.9353,0.0100,non-stationary
6,EUR_MRU,log-return,ADF,-35.7522,0.0000,stationary
7,EUR_MRU,log-return,KPSS,0.0884,0.1000,stationary


Levels are non-stationary, log-returns are stationary — both tests agree on both. **This is why every model in the project predicts a log *return* and anchors it to the last observed level, never the level directly.** A tree-based model trained on levels could never forecast above the highest rate it saw in training.

## 2. Volatility

In [3]:
window = config["evaluation"]["volatility_regimes"]["window"]
volatility = eda.rolling_volatility(returns[TARGETS], window)
display(volatility.describe().round(3))
viz.volatility_chart(volatility, window=window)

,USD_MRU,EUR_MRU
count,2038.000,2038.000
mean,6.086,8.980
std,3.966,3.360
min,0.531,3.496
25%,2.716,6.411
50%,5.883,8.336
75%,8.702,11.010
max,17.034,19.203


In [4]:
viz.returns_histogram(returns[TARGETS])

## 3. Autocorrelation — the one real signal

This is the most important chart in the project.

In [5]:
acf_table = eda.autocorrelation(returns["USD_MRU"], nlags=30)
display(acf_table.head(8).round(4))
viz.acf_chart(acf_table, column="acf")

,acf,pacf,conf_lower,conf_upper
lag,,,,
0,1.0000,1.0000,-0.0433,0.0433
1,-0.3023,-0.3025,-0.0433,0.0433
2,-0.0868,-0.1963,-0.0433,0.0433
3,0.0056,-0.0976,-0.0433,0.0433
4,0.0506,0.0037,-0.0433,0.0433
5,-0.0450,-0.0367,-0.0433,0.0433
6,-0.0021,-0.0218,-0.0433,0.0433
7,0.0439,0.0321,-0.0433,0.0433


In [6]:
viz.acf_chart(acf_table.dropna(subset=['pacf']), column='pacf')

**ACF(1) ≈ −0.30**: a move one day is partly given back the next. That is genuine mean reversion, and it is what the gradient-boosted model has to work with at short horizons.

**Caveat, stated plainly.** Part of this is an artefact rather than economics. Forward-filling closed days creates a flat day followed by a catch-up move, which induces negative autocorrelation on its own; the provider's own quote noise does the same. The honest reading is that *some* of the lag-1 signal is real and some is measurement — which is precisely why the value of the model is judged on a backtest, not on this chart.

## 4. Seasonality

In [7]:
dow = eda.seasonality_table(returns["USD_MRU"], "day_of_week")
display(dow.round(4))
viz.seasonality_chart(dow, title="USD/MRU: mean daily log-return by weekday (± 1 s.e.)")

,mean_pct,std_pct,n,t_stat
day_of_week,,,,
Mon,-0.0111,0.5076,411,-0.4443
Tue,0.0528,0.4644,410,2.3042
Wed,-0.0130,0.4891,410,-0.5373
Thu,0.0012,0.4225,410,0.0592
Fri,-0.0024,0.3687,411,-0.1301


In [8]:
month = eda.seasonality_table(returns["USD_MRU"], "month")
viz.seasonality_chart(month, title="USD/MRU: mean daily log-return by month (± 1 s.e.)")

In [9]:
print(f"largest |t| across the 5 weekdays: {dow['t_stat'].abs().max():.2f}")
print("With 5 simultaneous comparisons, |t| near 2 is what random data produces.")
print("Nothing here supports a tradeable day-of-week effect.")

largest |t| across the 5 weekdays: 2.30
With 5 simultaneous comparisons, |t| near 2 is what random data produces.
Nothing here supports a tradeable day-of-week effect.


## 5. Correlation

Correlations are computed on **returns**, never on levels: two trending level series correlate strongly for no reason other than both trending.

In [10]:
viz.correlation_heatmap(eda.correlation_matrix(returns))

In [11]:
others = [c for c in returns.columns if c != "USD_MRU"]
eda.lagged_correlation(returns, "USD_MRU", others, max_lag=5).round(3)

,BRENT_OIL,EUR_MRU,EUR_USD,GOLD
lag,,,,
0,-0.018,0.672,-0.113,-0.039
1,-0.003,-0.181,0.062,0.057
2,0.013,-0.033,0.042,0.013
3,0.031,0.006,0.003,-0.010
4,0.017,0.051,0.017,0.006
5,-0.025,-0.056,-0.028,0.006


Only non-negative lags are shown: a regressor is usable at forecast time only if it is already known. EUR/MRU correlates with USD/MRU by construction (it is the derived cross); the commodity series carry almost nothing.

## 6. Discrete policy steps

In [12]:
steps = eda.detect_step_events(frame["USD_MRU"], 1.0)
years = (frame.index[-1] - frame.index[0]).days / 365.25
print(f"{len(steps)} moves of 1% or more over {years:.1f} years ({len(steps)/years:.1f} per year)")
steps.tail(10).round(3)

65 moves of 1% or more over 7.9 years (8.3 per year)


,date,change_pct,level_before,level_after
55,2025-11-25,1.118,38.959,39.394
56,2026-01-27,1.398,39.036,39.581
57,2026-02-09,-1.268,39.326,38.827
58,2026-02-27,1.211,39.130,39.604
59,2026-03-26,1.000,39.203,39.595
60,2026-04-02,-1.355,39.567,39.031
61,2026-04-09,1.199,39.066,39.534
62,2026-04-29,-1.201,39.542,39.067
63,2026-05-18,-1.145,39.549,39.096
64,2026-06-02,1.094,39.389,39.820
